In [1]:
import torch_openreml
print(torch_openreml.example_data.john_alpha)

    plot rep block  gen   yield  row  col
0      1  R1    B1  G11  4.1172    1    1
1      2  R1    B1  G04  4.4461    2    1
2      3  R1    B1  G05  5.8757    3    1
3      4  R1    B1  G22  4.5784    4    1
4      5  R1    B2  G21  4.6540    5    1
..   ...  ..   ...  ...     ...  ...  ...
67    68  R3    B5  G24  3.5655   68    1
68    69  R3    B6  G03  2.8873   69    1
69    70  R3    B6  G05  4.1972   70    1
70    71  R3    B6  G20  3.7349   71    1
71    72  R3    B6  G07  3.6096   72    1

[72 rows x 7 columns]


In [2]:
import torch
import pandas as pd
from torch_openreml import MarginalREML, blup
from torch_openreml.covariance import DummyMatrix, IdentityMatrix, ScalarMatrix, Sum, CovariancePropagation, KroneckerProduct, Augment, BlockDiagonal
from torch_openreml.example_data import john_alpha

In [3]:
rep = john_alpha["rep"]
block = john_alpha["block"]
gen = john_alpha["gen"]
n = len(john_alpha)

y = torch.tensor(john_alpha["yield"].values)
X = Augment(torch.ones(n, 1), DummyMatrix(rep, drop_first=True))()

V = Sum(
    random = CovariancePropagation(
        Z = Augment(
            Z_gen = DummyMatrix(gen),
            Z_rep_block = DummyMatrix(rep, block),
        ),
        G = BlockDiagonal(
            G_gen = ScalarMatrix(gen),
            G_rep_block = KroneckerProduct(
                G_rep = IdentityMatrix(rep),
                G_block = ScalarMatrix(block),
            ),
        ),
    ),
    residual = ScalarMatrix(n),
)

print(V)

Sum(
  operands={
    'random': CovariancePropagation(
      operands={
        'Z': Augment(
          operands={
            'Z_gen': DummyMatrix(shape=(72, 24)),
            'Z_rep_block': DummyMatrix(shape=(72, 18))
          }
        ),
        'G': BlockDiagonal(
          operands={
            'G_gen': ScalarMatrix(shape=(24, 24), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}}),
            'G_rep_block': KroneckerProduct(
              operands={
                'G_rep': IdentityMatrix(shape=(3, 3)),
                'G_block': ScalarMatrix(shape=(6, 6), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}})
              }
            )
          }
        )
      }
    ),
    'residual': ScalarMatrix(shape=(72, 72), param_specs={'sigma^2': {'fixed': False, 'default': tensor([0.]), 'trans': TransformExpPow2()}})
  }
)


In [4]:
reml = MarginalREML(V)
theta_hat, beta_hat, n_iter = reml.optimize(y, X, torch.zeros(3), verbose=1)
print(theta_hat, V.build_params(theta_hat))
print(V.free_param_names)
print(beta_hat)

[  1] ∥∇∥:      41.8197, ∥Δ∥: 8.9438, η: 1.00, ∥Δᶜ∥: 8.9438, log 𝓛: -34.3129
[  2] ∥∇∥:  315236.5000, ∥Δ∥: 0.8838, η: 1.00, ∥Δᶜ∥: 0.8838, log 𝓛: -201188.2344 (-201153.9215)
[  3] ∥∇∥:  119229.1953, ∥Δ∥: 0.8520, η: 1.00, ∥Δᶜ∥: 0.8520, log 𝓛: -71844.9922 (+129343.2422)
[  4] ∥∇∥:   44277.3672, ∥Δ∥: 0.8143, η: 1.00, ∥Δᶜ∥: 0.8143, log 𝓛: -26066.7031 (+45778.2891)
[  5] ∥∇∥:   16331.1357, ∥Δ∥: 0.7587, η: 1.00, ∥Δᶜ∥: 0.7587, log 𝓛: -9457.5859 (+16609.1172)
[  6] ∥∇∥:    6001.6343, ∥Δ∥: 0.7142, η: 1.00, ∥Δᶜ∥: 0.7142, log 𝓛: -3393.0317 (+6064.5542)
[  7] ∥∇∥:    2194.9788, ∥Δ∥: 0.7031, η: 1.00, ∥Δᶜ∥: 0.7031, log 𝓛: -1181.3386 (+2211.6931)
[  8] ∥∇∥:     793.8171, ∥Δ∥: 0.6952, η: 1.00, ∥Δᶜ∥: 0.6952, log 𝓛: -382.5117 (+798.8269)
[  9] ∥∇∥:     278.6996, ∥Δ∥: 0.6714, η: 1.00, ∥Δᶜ∥: 0.6714, log 𝓛: -102.3060 (+280.2057)
[ 10] ∥∇∥:      90.4162, ∥Δ∥: 0.5957, η: 1.00, ∥Δᶜ∥: 0.5957, log 𝓛: -11.4293 (+90.8767)
[ 11] ∥∇∥:      23.9234, ∥Δ∥: 0.4006, η: 1.00, ∥Δᶜ∥: 0.4006, log 𝓛:  12.6791 (+24.1085)
[ 12]

In [5]:
tree = reml.tree()

In [6]:
b_hat = blup(y, X, tree["random/Z"], tree["random/G"], tree["/"])
b_hat

tensor([ 0.5012,  0.0050, -0.7846,  0.0061,  0.4749,  0.0446, -0.3089,  0.0622,
        -0.8099, -0.0894, -0.1964,  0.2258,  0.2317,  0.2434,  0.4247,  0.2010,
         0.0781, -0.1102,  0.2896, -0.3390,  0.2561,  0.0241, -0.1770, -0.2531,
         0.1231, -0.1412, -0.1504, -0.1068,  0.0737,  0.2015, -0.5326, -0.3012,
         0.2432,  0.1349,  0.2753,  0.1804,  0.0506, -0.0478,  0.1511,  0.0538,
        -0.0080, -0.1996])

In [7]:
gen_blup = blup(y, X, tree["random/Z/Z_gen"], tree["random/G/G_gen"], tree["/"])
pd.Series(gen_blup, index=sorted(set(gen)))

G01    0.501183
G02    0.004962
G03   -0.784563
G04    0.006125
G05    0.474950
G06    0.044640
G07   -0.308948
G08    0.062229
G09   -0.809932
G10   -0.089373
G11   -0.196435
G12    0.225758
G13    0.231665
G14    0.243399
G15    0.424699
G16    0.200964
G17    0.078078
G18   -0.110181
G19    0.289576
G20   -0.338969
G21    0.256132
G22    0.024089
G23   -0.176998
G24   -0.253058
dtype: float32